# 03 — LSTM forecasting (PyTorch)

Huấn luyện LSTM dự báo doanh số ngày tiếp theo từ **30 ngày lịch sử liên tiếp**. Input features dựa trên `src/features.py`: Sales lịch sử, trạng thái cửa hàng/khuyến mãi/ngày nghỉ và mã hóa chu kỳ lịch.

**Chống leakage:** scaler chỉ fit trên train; sequence chỉ dùng 30 ngày trước target; validation và holdout được dự báo theo thứ tự thời gian. Với notebook này, mỗi dự báo là one-step-ahead và dùng doanh số thực tế đã quan sát của các ngày trước đó — phù hợp đánh giá rolling-origin, không phải dự báo nhiều ngày hoàn toàn tự hồi quy.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = ARTIFACT_DIR / "results"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
for folder in (RESULT_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

LOOKBACK = 30
STORE_ID = 1

from src.data import load_rossmann_data, chronological_split
from src.features import (
    create_calendar_features, prepare_features, get_feature_columns,
    fit_feature_scaler, transform_features, fit_target_scaler,
    transform_target, inverse_transform_target, make_sequences,
    DEFAULT_LOOKBACK
)

train_df, official_test_df = load_rossmann_data(DATA_DIR, store_id=STORE_ID)
train_df = train_df.sort_values("Date").reset_index(drop=True)
splits = chronological_split(train_df, train_ratio=0.70, val_ratio=0.15, holdout_ratio=0.15)
print(f"Store: {STORE_ID} | labeled rows: {len(train_df):,}")
print(f"Train:      {splits['train']['Date'].min().date()} -> {splits['train']['Date'].max().date()} ({len(splits['train'])})")
print(f"Validation: {splits['validation']['Date'].min().date()} -> {splits['validation']['Date'].max().date()} ({len(splits['validation'])})")
print(f"Holdout:    {splits['holdout']['Date'].min().date()} -> {splits['holdout']['Date'].max().date()} ({len(splits['holdout'])})")

## 1. Chuẩn bị features và sequences

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

feature_df = prepare_features(train_df, include_lags=False)
feature_cols = get_feature_columns(include_lags=False)

train_end = len(splits["train"])
val_end = train_end + len(splits["validation"])
holdout_start_idx = val_end

# Fit scalers on train partition only.
feature_scaler = fit_feature_scaler(feature_df.iloc[:train_end], feature_cols)
target_scaler = fit_target_scaler(train_df.iloc[:train_end])

feature_array = transform_features(feature_df, feature_scaler, feature_cols)
target_scaled = np.full(len(train_df), np.nan, dtype=np.float32)
target_scaled[:] = target_scaler.transform(train_df[["Sales"]]).astype(np.float32).ravel()
dates = train_df["Date"]

train_idx = np.arange(0, train_end)
val_idx = np.arange(train_end, val_end)
holdout_idx = np.arange(val_end, len(train_df))

X_train, y_train, idx_train = make_sequences(feature_array, target_scaled, dates, train_idx, lookback=LOOKBACK)
X_val, y_val, idx_val = make_sequences(feature_array, target_scaled, dates, val_idx, lookback=LOOKBACK)
X_hold, y_hold, idx_hold = make_sequences(feature_array, target_scaled, dates, holdout_idx, lookback=LOOKBACK)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_holdout:", X_hold.shape)
assert X_train.shape[1] == 30
assert X_train.shape[2] == len(feature_cols)
if len(X_train) == 0 or len(X_val) == 0 or len(X_hold) == 0:
    raise ValueError("No sequences created. Check date continuity, data size, and feature columns.")

## 2. Định nghĩa mô hình LSTM

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        return self.head(sequence_output[:, -1, :])

model = LSTMForecaster(input_size=len(feature_cols)).to(DEVICE)
print(model)

## 3. Huấn luyện với early stopping

In [ ]:
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 1e-3
PATIENCE = 8

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE, shuffle=False
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"train_loss": [], "val_loss": []}
best_val = float("inf")
best_state = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_losses.append(criterion(model(xb), yb).item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f}")

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)

## 4. Loss curve và đánh giá holdout

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.xlabel("Epoch"); plt.ylabel("MSE loss"); plt.title("LSTM training history")
plt.legend(); plt.tight_layout(); plt.show()

model.eval()
with torch.no_grad():
    hold_pred_scaled = model(torch.from_numpy(X_hold).to(DEVICE)).cpu().numpy().reshape(-1, 1)

hold_pred = inverse_transform_target(hold_pred_scaled, target_scaler)
hold_actual = train_df.loc[idx_hold, "Sales"].to_numpy(dtype=float)
hold_dates = train_df.loc[idx_hold, "Date"].to_numpy()

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae = np.mean(np.abs(y_true-y_pred))
    rmse = np.sqrt(np.mean((y_true-y_pred)**2))
    denom = np.sum(np.abs(y_true))
    wape = 100*np.sum(np.abs(y_true-y_pred))/denom if denom else np.nan
    return {"MAE": mae, "RMSE": rmse, "WAPE_pct": wape}

lstm_metrics = regression_metrics(hold_actual, hold_pred)
display(pd.DataFrame([{"model":"LSTM", **lstm_metrics}]))

## 5. Vẽ dự báo và lưu artifact

In [ ]:
lstm_results = pd.DataFrame({
    "Date": pd.to_datetime(hold_dates),
    "Store": STORE_ID,
    "Sales": hold_actual,
    "LSTM_Prediction": hold_pred.ravel(),
})
plt.figure(figsize=(14, 5))
plt.plot(lstm_results["Date"], lstm_results["Sales"], label="Actual", linewidth=2)
plt.plot(lstm_results["Date"], lstm_results["LSTM_Prediction"], label="LSTM forecast")
plt.title("Rossmann Store 1 — LSTM holdout predictions")
plt.xlabel("Date"); plt.ylabel("Sales"); plt.legend(); plt.tight_layout(); plt.show()

lstm_results.to_csv(RESULT_DIR / "lstm_holdout_predictions.csv", index=False)
pd.DataFrame([{"model":"LSTM", **lstm_metrics}]).to_csv(RESULT_DIR / "lstm_metrics.csv", index=False)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "input_size": len(feature_cols),
    "feature_columns": feature_cols,
    "lookback": LOOKBACK,
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "best_val_loss": best_val,
    "store_id": STORE_ID,
}
torch.save(checkpoint, CHECKPOINT_DIR / "lstm_store1.pt")
import joblib
joblib.dump(feature_scaler, CHECKPOINT_DIR / "feature_scaler_store1.joblib")
joblib.dump(target_scaler, CHECKPOINT_DIR / "target_scaler_store1.joblib")
print("Saved predictions and checkpoint under", ARTIFACT_DIR)

### Lưu ý phương pháp
- `Sales` trong cửa sổ input chỉ là lịch sử đến ngày trước target; không đưa Sales cùng ngày target vào input.
- Notebook giả định một dòng mỗi ngày liên tục. Hàm `make_sequences` sẽ loại sequence nếu thiếu ngày.
- Đây là dự báo one-step-ahead trên holdout với dữ liệu Sales thực tế các ngày đã qua. Nếu mục tiêu là dự báo 6 tuần tương lai mà không biết Sales thực tế, cần thiết kế recursive/multi-step forecasting riêng.